**02_linear_regression_Homework**

In [12]:
import pandas as pd
import numpy as np

# Load the dataset
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [13]:
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [14]:
# Filters as instructed
cols = [
    'engine_displacement', 
    'horsepower', 
    'vehicle_weight', 
    'model_year', 
    'fuel_efficiency_mpg'
]
df = df[cols]

In [15]:
df

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0
...,...,...,...,...,...
9995,2470,271.0,4690,2005,27.4
9996,2300,244.0,4370,1993,30.0
9997,2480,267.0,4590,2014,28.1
9998,2180,270.0,4450,2004,32.2


In [16]:
#Question 1: Missing values
print(df.isnull().sum())

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


In [17]:
#Question 2: Median for horsepower
median_hp = df['horsepower'].median()
print(f"Median horsepower: {median_hp}")

Median horsepower: 254.0


In [18]:
def train_linear_regression(X, y, r=0.0):
    # Add dummy column for the bias term
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    
    # Calculate X^T X
    XTX = X.T.dot(X)
    
    # Add regularization to the diagonal
    XTX = XTX + r * np.eye(XTX.shape[0])
    
    # Inverse of (X^T X) multiplied by X^T y
    weights = np.linalg.inv(XTX).dot(X.T).dot(y)
    
    return weights[0], weights[1:]

def rmse(y, y_pred):
    error = y - y_pred
    mse = (error ** 2).mean()
    return np.sqrt(mse)

In [6]:
#Question 3: Filling NAs (0 vs Mean)
# Shuffle and split (Seed 42)
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val = df.iloc[idx[n_train:n_train + n_val]].copy()

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']

# Option 1: Fill with 0
X_train_0 = df_train.fillna(0).values
X_val_0 = df_val.fillna(0).values

w0_0, w_0 = train_linear_regression(X_train_0, y_train)
y_pred_0 = w0_0 + X_val_0.dot(w_0)
rmse_0 = round(rmse(y_val, y_pred_0), 3)

# Option 2: Fill with Mean
mean_hp = df_train['horsepower'].mean()  # Computed ONLY on training set
X_train_mean = df_train.fillna(mean_hp).values
X_val_mean = df_val.fillna(mean_hp).values

w0_mean, w_mean = train_linear_regression(X_train_mean, y_train)
y_pred_mean = w0_mean + X_val_mean.dot(w_mean)
rmse_mean = round(rmse(y_val, y_pred_mean), 3)

print(f"RMSE (Fill 0): {rmse_0}")
print(f"RMSE (Fill Mean): {rmse_mean}")

RMSE (Fill 0): 2.205
RMSE (Fill Mean): 2.202


In [7]:
#Question 4: Best Regularization
r_values = [0, 0.01, 0.1, 1, 5, 10, 100]

for r in r_values:
    w0, w = train_linear_regression(X_train_0, y_train, r=r)
    y_pred = w0 + X_val_0.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:<5} | RMSE = {score}")


r = 0     | RMSE = 2.2053
r = 0.01  | RMSE = 2.2058
r = 0.1   | RMSE = 2.2241
r = 1     | RMSE = 2.3492
r = 5     | RMSE = 2.4094
r = 10    | RMSE = 2.4195
r = 100   | RMSE = 2.4292


In [8]:
#Question 5: RMSE Standard Deviation
seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_scores = []

for s in seeds:
    np.random.seed(s)
    idx_s = np.arange(n)
    np.random.shuffle(idx_s)
    
    df_train_s = df.iloc[idx_s[:n_train]].copy()
    df_val_s = df.iloc[idx_s[n_train:n_train + n_val]].copy()
    
    y_train_s = df_train_s['fuel_efficiency_mpg'].values
    y_val_s = df_val_s['fuel_efficiency_mpg'].values
    
    X_train_s = df_train_s.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
    X_val_s = df_val_s.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
    
    w0_s, w_s = train_linear_regression(X_train_s, y_train_s)
    y_pred_s = w0_s + X_val_s.dot(w_s)
    
    rmse_scores.append(rmse(y_val_s, y_pred_s))

std_dev = round(np.std(rmse_scores), 3)
print(f"Standard Deviation of RMSE scores: {std_dev}")

Standard Deviation of RMSE scores: 0.029


In [9]:
#Question 6: Evaluation on Test Data
np.random.seed(9)
idx_9 = np.arange(n)
np.random.shuffle(idx_9)

df_train_9 = df.iloc[idx_9[:n_train]].copy()
df_val_9 = df.iloc[idx_9[n_train:n_train + n_val]].copy()
df_test_9 = df.iloc[idx_9[n_train + n_val:]].copy()

# Combine train and val
df_full_train = pd.concat([df_train_9, df_val_9])

y_full_train = df_full_train['fuel_efficiency_mpg'].values
y_test_9 = df_test_9['fuel_efficiency_mpg'].values

X_full_train = df_full_train.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
X_test_9 = df_test_9.drop('fuel_efficiency_mpg', axis=1).fillna(0).values

# Train with r=0.001
w0_9, w_9 = train_linear_regression(X_full_train, y_full_train, r=0.001)
y_pred_test = w0_9 + X_test_9.dot(w_9)

test_rmse = round(rmse(y_test_9, y_pred_test), 3)
print(f"Test RMSE: {test_rmse}")

Test RMSE: 2.236
